# Generar los datos sintéticos

Corre `data/generar_datos.py`: simula primero la "verdad" (qué cliente existe, en qué
plan, cuándo entra y cuándo se va) y **solo después** la deforma para producir los
archivos de origen. La verdad queda guardada en `data/control_totales.json` — sirve
para verificar en el notebook 03 que el pipeline reconstruye los mismos números.

In [1]:
import subprocess, sys
resultado = subprocess.run(
    [sys.executable, "data/generar_datos.py"],
    cwd=r"D:\Claudia\Alegra", capture_output=True, text=True
)
print(resultado.stdout)
if resultado.returncode != 0:
    print(resultado.stderr)
    raise RuntimeError("generar_datos.py fallo")

Control de verdad calculado: 180 clientes de pago, 17 casos dificiles de rebranding.
MRR verdad diciembre 2024: 57,775,536 COP

Archivos generados:
  CRM:        262 filas -> data/raw/crm/crm_hubspot_export.csv
  Billing:    286 filas -> data/raw/billing/billing_stripe_export.csv
  Contratos:  49 filas -> data/raw/contratos/contratos.xlsx
  Marketing:  72 filas -> data/raw/marketing/gasto_canales.csv
  Seeds:      planes.csv, canales.csv, tasa_cambio.csv
  Control:    data/control_totales.json



## Muestra de cada archivo de origen — la mugre es intencional

In [2]:
import pandas as pd
pd.set_option("display.max_colwidth", 40)
ROOT = r"D:\Claudia\Alegra"

crm = pd.read_csv(f"{ROOT}/data/raw/crm/crm_hubspot_export.csv")
print("CRM — nótese el formato de fecha mixto (ISO y DD/MM/AAAA) y el canal en texto libre:")
crm[["nombre_empresa", "canal_adquisicion", "fecha_alta_lead", "etapa"]].sample(6, random_state=7)

CRM — nótese el formato de fecha mixto (ISO y DD/MM/AAAA) y el canal en texto libre:


,nombre_empresa,canal_adquisicion,fecha_alta_lead,etapa
207,"BURBANO, REYES AND CASTRO",organico,02/07/2024,Cliente
22,"ACEVEDO, LOZANO AND VARGAS",SEO,2024-04-19,Cliente
132,"González, Villamizar and Vargas",meta ads,22/04/2024,Cliente
102,Rodríguez PLC,SEO,2024-07-10,Cliente
57,"MARTÍNEZ, ARANGO AND JAIMES",Referral,17/01/2024,Cliente
168,"Velasco, Fernández and Perdomo",Partner,2024-03-20,Lead


In [3]:
billing = pd.read_csv(f"{ROOT}/data/raw/billing/billing_stripe_export.csv")
print("Billing — el mismo plan escrito de varias formas, y montos con formato de miles:")
sucios = billing[pd.to_numeric(billing["monto"], errors="coerce").isna()]
print(f"\n{len(sucios)} filas con monto en formato no directo (ej. '$149.000'):")
sucios[["nombre_cliente_facturacion", "plan_contratado", "monto", "moneda"]]

Billing — el mismo plan escrito de varias formas, y montos con formato de miles:

3 filas con monto en formato no directo (ej. '$149.000'):


,nombre_cliente_facturacion,plan_contratado,monto,moneda
88,Mosquera-Peña,STARTER,$149.000,COP
106,Leal Inc,STARTER,$149.000,COP
167,Bedoya-Castillo,Starter,$149.000,COP


In [4]:
contratos = pd.read_excel(f"{ROOT}/data/raw/contratos/contratos.xlsx")
print("Contratos — la misma columna de fecha mezcla fecha real y texto DD/MM/AAAA:")
print(contratos["fecha_firma"].apply(type).value_counts())
contratos[["razon_social", "plan_contratado", "fecha_firma"]].head(6)

Contratos — la misma columna de fecha mezcla fecha real y texto DD/MM/AAAA:
fecha_firma
<class 'datetime.datetime'>    25
<class 'str'>                  24
Name: count, dtype: int64


,razon_social,plan_contratado,fecha_firma
0,GALVIS LTD SAS,Pro,2024-05-04 00:00:00
1,"VÁSQUEZ, TORRES AND FERNÁNDEZ SAS",Business,22/02/2024
2,SANABRIA-AGUDELO S.A.S.,Business,2024-01-11 00:00:00
3,GÓMEZ AND SONS SAS,Pro,2024-08-18 00:00:00
4,RIVERA GROUP S.A.S.,Pro,2024-09-15 00:00:00
5,"PACHECO, TORRES AND VALENCIA S.A.S.",Pro,2024-09-16 00:00:00


## Catálogo de mugre sembrada

| # | Problema | Dónde |
|---|---|---|
| 1 | Mismo cliente, ID distinto en cada sistema | CRM / Billing / Contratos |
| 2 | Nombre de empresa escrito distinto entre sistemas | Los 3 |
| 3 | Teléfonos/emails con formato inconsistente | CRM |
| 4 | Fechas en 3 formatos (ISO, DD/MM/AAAA, fecha nativa de Excel) | CRM / Contratos |
| 5 | Moneda mixta COP/USD | Billing / Contratos |
| 6 | Alias de plan ("Pro", "PRO-mensual", "pro_monthly") | CRM / Billing |
| 7 | Alias de canal ("Google Ads", "SEM", "google ads") | CRM / Marketing |
| 8 | Leads/trials que nunca convierten, mezclados con clientes de pago | CRM |
| 9 | Filas duplicadas por reintento de sincronización | CRM |
| 10 | ~8% de empresas cambian de razón social a mitad de año | CRM / Billing |
| 11 | Nombres de empresa que colisionan por azar entre dos clientes reales | Billing |
| 12 | Eventos con tipo perdido (van a cuarentena) | Billing |

In [5]:
import json
control = json.load(open(f"{ROOT}/data/control_totales.json", encoding="utf-8"))
print(f"Clientes de pago (verdad): {control['num_clientes_pagos_verdad']}")
print(f"Leads/trial (verdad): {control['num_leads_trial_verdad']}")
print(f"Casos dificiles de rebranding (verdad): {control['num_casos_dificiles_rebrand']}")

df_control = pd.DataFrame(control["mensual"]).T
df_control.index.name = "id_periodo"
df_control[["mrr_cop", "clientes_activos", "altas", "bajas"]]

Clientes de pago (verdad): 180
Leads/trial (verdad): 55
Casos dificiles de rebranding (verdad): 17


,mrr_cop,clientes_activos,altas,bajas
id_periodo,,,,
202401,8254340.0,28.0,28.0,0.0
202402,12928290.0,44.0,16.0,0.0
202403,21759892.0,69.0,26.0,1.0
202404,25161018.0,82.0,16.0,3.0
202405,34272630.0,104.0,24.0,2.0
202406,40817680.0,115.0,13.0,2.0
202407,46899756.0,129.0,18.0,4.0
202408,55535720.0,145.0,18.0,2.0
202409,60513300.0,165.0,22.0,2.0
